In [6]:
import pandas as pd
import numpy as np

# Load Carvana data
df = pd.read_csv('../data/raw/carvana.csv')

# Create target: premium = 1 if Price >= 30000, else 0
df['premium'] = (df['Price'] >= 30000).astype(int)

# Check class balance
print(df['premium'].value_counts())
print(df['premium'].value_counts(normalize=True))

premium
0    20699
1     1301
Name: count, dtype: int64
premium
0    0.940864
1    0.059136
Name: proportion, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

# Features and target
X = df[['Year', 'Miles']]
y = df['Price']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 17600
Testing samples: 4400


In [10]:
from sklearn.linear_model import LogisticRegression
# or: from sklearn.ensemble import RandomForestClassifier

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

c:\Users\user\Desktop\klab-ai-marieclaire\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
from sklearn.model_selection import train_test_split

# Features and target
X = df[['Year', 'Miles']]  # Features
y = df['premium']          # ✅ USE PREMIUM, NOT PRICE

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 17600
Testing samples: 4400


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix)

# Train the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate all metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='binary')
rec = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')
auc = roc_auc_score(y_test, y_pred_proba)

# Print confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("=== CONFUSION MATRIX ===")
print(cm)
print(f"\nTrue Negatives (TN):  {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP):  {cm[1,1]}")

# Print all metrics
print("\n=== EVALUATION METRICS ===")
print(f"Accuracy:  {acc:.3f}  ← Overall correctness (MISLEADING on imbalanced data)")
print(f"Precision: {prec:.3f}  ← When it says premium, how often right?")
print(f"Recall:    {rec:.3f}  ← Of real premiums, how many did it find?")
print(f"F1:        {f1:.3f}  ← Balance of precision & recall")
print(f"ROC-AUC:   {auc:.3f}  ← Overall ranking quality")

=== CONFUSION MATRIX ===
[[4139    1]
 [ 256    4]]

True Negatives (TN):  4139
False Positives (FP): 1
False Negatives (FN): 256
True Positives (TP):  4

=== EVALUATION METRICS ===
Accuracy:  0.942  ← Overall correctness (MISLEADING on imbalanced data)
Precision: 0.800  ← When it says premium, how often right?
Recall:    0.015  ← Of real premiums, how many did it find?
F1:        0.030  ← Balance of precision & recall
ROC-AUC:   0.804  ← Overall ranking quality
